In [41]:
import re
from pathlib import Path
import pandas as pd

# Файлы

INPUT_PANEL = Path("panel_sorted.bed")
INPUT_EXONS = Path("panel_exons.tsv")
INPUT_REFSEQ = Path("panel_refseq_full.tsv")
INPUT_CLOSEST = Path("closest_check.tsv")

OUTPUT_EXONIC = Path("panel_annotated.tsv")
OUTPUT_NO_EXON = Path("no_exon_annotated.tsv")
OUTPUT_FINAL = Path("panel_fully_annotated.tsv")

OUTPUT_NO_EXON_TARGETS = Path("targets_without_exon.tsv")
OUTPUT_NO_EXON_ANNOTATION = Path("targets_without_exon_annotation.tsv")


# Структура входных файлов

TARGET_COLUMNS = [
    "p_chrom",
    "p_start",
    "p_end",
    "p_name",
]

BED_COLUMNS = [
    "p_chrom",
    "p_start",
    "p_end",
    "p_name",
    "p_score",
    "p_attrs",
]

EXON_COLUMNS = BED_COLUMNS + [
    "e_chrom",
    "e_source",
    "e_feature",
    "e_start",
    "e_end",
    "e_score",
    "e_strand",
    "e_frame",
    "e_attrs",
]

REFSEQ_COLUMNS = BED_COLUMNS + [
    "g_chrom",
    "g_source",
    "g_feature",
    "g_start",
    "g_end",
    "g_score",
    "g_strand",
    "g_frame",
    "g_attrs",
]

CLOSEST_COLUMNS = [
    "a_chrom",
    "a_start",
    "a_end",
    "a_name",
    "e_chrom",
    "e_start",
    "e_end",
    "e_attrs",
    "distance",
]


# Константы

NA_VALUE = "NA"
NO_EXON_VALUE = "."

In [42]:
def read_table(path, columns):
    """Читает таблицу без заголовка с заданными именами столбцов."""
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=columns,
        dtype=str,
    )

    if df.empty:
        raise ValueError(f"Файл пуст: {path}")

    return df


def unique_values_as_string(series):
    """Возвращает уникальные непустые значения через ';'."""
    values = sorted(
        {
            value
            for value in series
            if value != NA_VALUE
        }
    )

    return ";".join(values) if values else NA_VALUE


def calculate_overlap(df):
    """Рассчитывает длину пересечения BED-региона и GFF exon."""
    panel_start = df["p_start"].astype(int)
    panel_end = df["p_end"].astype(int)

    exon_start = df["e_start"].astype(int) - 1
    exon_end = df["e_end"].astype(int)

    return (
        pd.concat([panel_end, exon_end], axis=1).min(axis=1)
        - pd.concat([panel_start, exon_start], axis=1).max(axis=1)
    ).clip(lower=0)


def classify_annotation(features):
    """Определяет тип RefSeq-аннотации."""
    features = set(features.split(";"))

    if "transcriptional_cis_regulatory_region" in features:
        return "regulatory"

    if features & {"gene", "mRNA", "transcript"}:
        return "gene_or_transcript_region"

    if "match" in features:
        return "RefSeq_match"

    if "biological_region" in features:
        return "biological_region"

    if features == {"region"}:
        return "region_only"

    return "other"


def get_annotation_label(row):
    """Возвращает название гена или запасной идентификатор."""
    if row["gene"] != NA_VALUE:
        return row["gene"]

    if row["GeneID"] != NA_VALUE:
        return f"GeneID{row['GeneID']}"

    return "unassigned"


def make_exon_id(row):
    """Формирует идентификатор для target, пересекающего exon."""
    if row["gene"] == NA_VALUE:
        return f"unassigned_exon{row['exon_num']}"

    if row["exon_num"] == NA_VALUE:
        return f"{row['gene']}_exon"

    return f"{row['gene']}_exon{row['exon_num']}"


def make_non_exon_id(row):
    """Формирует идентификатор для target без пересечения с exon."""
    distance = row["nearest_exon_distance"]
    label = get_annotation_label(row)
    status = row["annotation_status"]

    if status == "regulatory":
        return f"{label}_regulatory_dist{distance}bp"

    if status == "gene_or_transcript_region":
        return f"{label}_dist{distance}bp"

    if status == "RefSeq_match":
        return f"unassigned_match_dist{distance}bp"

    if status == "biological_region":
        return f"{label}_biological_region_dist{distance}bp"

    if status == "region_only":
        return f"intergenic_dist{distance}bp"

    return f"unclassified_{status}_dist{distance}bp"

In [43]:
# Чтение исходной панели и результатов bedtools.

original_panel = read_table(
    INPUT_PANEL,
    BED_COLUMNS,
)

exons = read_table(
    INPUT_EXONS,
    EXON_COLUMNS,
)

refseq = read_table(
    INPUT_REFSEQ,
    REFSEQ_COLUMNS,
)

closest = read_table(
    INPUT_CLOSEST,
    CLOSEST_COLUMNS,
)

In [44]:
# Шаблоны для извлечения аннотационных данных из GFF attributes.

GENE_PATTERN = re.compile(
    r"(?:^|;)gene=([^;]+)"
)

GENE_ID_PATTERN = re.compile(
    r"GeneID:(\d+)"
)

TRANSCRIPT_ID_PATTERN = re.compile(
    r"(?:^|;)transcript_id=([^;]+)"
)

PARENT_RNA_PATTERN = re.compile(
    r"(?:^|;)Parent=rna-([^;]+)"
)

EXON_ID_PATTERN = re.compile(
    r"(?:^|;)ID=exon-([^;]+)"
)


def parse_gff_attributes(attrs):
    """
    Извлекает аннотационные данные из поля GFF attributes.

    Возвращает:
        gene
        GeneID
        transcript
        exon_num
        is_mane
        is_refseq_select
    """

    if pd.isna(attrs) or attrs == ".":
        return {
            "gene": NA_VALUE,
            "GeneID": NA_VALUE,
            "transcript": NA_VALUE,
            "exon_num": NA_VALUE,
            "is_mane": False,
            "is_refseq_select": False,
        }

    # Название гена.
    gene_match = GENE_PATTERN.search(attrs)

    gene = (
        gene_match.group(1)
        if gene_match
        else NA_VALUE
    )

    # GeneID.
    gene_id_match = GENE_ID_PATTERN.search(attrs)

    gene_id = (
        gene_id_match.group(1)
        if gene_id_match
        else NA_VALUE
    )

    # Идентификатор транскрипта.
    transcript_match = TRANSCRIPT_ID_PATTERN.search(attrs)

    if transcript_match:
        transcript = transcript_match.group(1)
    else:
        parent_match = PARENT_RNA_PATTERN.search(attrs)

        transcript = (
            parent_match.group(1)
            if parent_match
            else NA_VALUE
        )

    # Номер экзона.
    exon_num = NA_VALUE

    exon_match = EXON_ID_PATTERN.search(attrs)

    if exon_match:
        exon_id = exon_match.group(1)

        if "-" in exon_id:
            _, exon_num = exon_id.rsplit("-", 1)

    return {
        "gene": gene,
        "GeneID": gene_id,
        "transcript": transcript,
        "exon_num": exon_num,
        "is_mane": "MANE Select" in attrs,
        "is_refseq_select": "RefSeq Select" in attrs,
    }

In [45]:
# Разделение target на exonic и non-exonic.

no_exon_mask = exons["e_chrom"] == NO_EXON_VALUE

no_exon_ids = set(
    exons.loc[
        no_exon_mask,
        "p_name",
    ]
)

print(
    f"Регионов без пересечения с exon: "
    f"{len(no_exon_ids):,}"
)

no_exon_targets = (
    exons.loc[
        no_exon_mask,
        TARGET_COLUMNS,
    ]
    .drop_duplicates()
    .copy()
)

no_exon_targets.to_csv(
    OUTPUT_NO_EXON_TARGETS,
    sep="\t",
    index=False,
)


exonic = exons.loc[
    ~no_exon_mask
].copy()

print(
    f"Строк с пересечением с exon: "
    f"{len(exonic):,}"
)

print(
    f"Target с пересечением с exon: "
    f"{exonic['p_name'].nunique():,}"
)


parsed_attributes = pd.DataFrame(
    exonic["e_attrs"]
    .apply(parse_gff_attributes)
    .tolist(),
    index=exonic.index,
)

exonic = pd.concat(
    [
        exonic,
        parsed_attributes,
    ],
    axis=1,
)

exonic["overlap"] = calculate_overlap(
    exonic
)

Регионов без пересечения с exon: 25
Строк с пересечением с exon: 1,322
Target с пересечением с exon: 361


In [46]:
# Выбор одной лучшей exon-аннотации для каждого target.
#
# Приоритет выбора:
# 1. MANE Select;
# 2. RefSeq Select;
# 3. максимальная длина пересечения target с exon.

best_exonic = (
    exonic
    .sort_values(
        by=[
            *TARGET_COLUMNS,
            "is_mane",
            "is_refseq_select",
            "overlap",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            False,
            False,
        ],
    )
    .drop_duplicates(
        subset=TARGET_COLUMNS,
        keep="first",
    )
    .copy()
)

# Контроль: после выбора должна остаться ровно одна аннотация для каждого exonic target.

if best_exonic["p_name"].duplicated().any():
    raise ValueError(
        "После выбора лучшего exon обнаружены "
        "дублирующиеся target."
    )

print(
    f"Уникальных exonic target: "
    f"{len(best_exonic):,}"
)

Уникальных exonic target: 361


In [47]:
# Формирование итогового идентификатора для exonic target вида: gene_exonN
# Если gene не определён, используется идентификатор GeneID или метка unassigned.

def make_exon_id(row):
    if row["gene"] != NA_VALUE:
        if row["exon_num"] != NA_VALUE:
            return f"{row['gene']}_exon{row['exon_num']}"

        return f"{row['gene']}_exon"

    if row["GeneID"] != NA_VALUE:
        if row["exon_num"] != NA_VALUE:
            return f"GeneID{row['GeneID']}_exon{row['exon_num']}"

        return f"GeneID{row['GeneID']}_exon"

    if row["exon_num"] != NA_VALUE:
        return f"unassigned_exon{row['exon_num']}"

    return "unassigned_exon"


best_exonic["final_id"] = best_exonic.apply(
    make_exon_id,
    axis=1,
)

best_exonic["annotation_status"] = "exon"
best_exonic["nearest_exon_distance"] = "0"


EXONIC_OUTPUT_COLUMNS = [
    "p_chrom",
    "p_start",
    "p_end",
    "p_name",
    "gene",
    "GeneID",
    "transcript",
    "exon_num",
    "annotation_status",
    "nearest_exon_distance",
    "final_id",
]

exonic_result = best_exonic[
    EXONIC_OUTPUT_COLUMNS
].copy()

exonic_result.to_csv(
    OUTPUT_EXONIC,
    sep="\t",
    index=False,
)

In [48]:
# Извлечение RefSeq-аннотаций для target без пересечения с exon.

no_exon_refseq = refseq.loc[
    refseq["p_name"].isin(no_exon_ids)
].copy()

refseq_attributes = pd.DataFrame(
    no_exon_refseq["g_attrs"]
    .apply(parse_gff_attributes)
    .tolist(),
    index=no_exon_refseq.index,
)

no_exon_refseq = pd.concat(
    [
        no_exon_refseq,
        refseq_attributes,
    ],
    axis=1,
)

annotation_summary = (
    no_exon_refseq
    .groupby("p_name")
    .agg(
        feature_types=(
            "g_feature",
            unique_values_as_string,
        ),
        gene=(
            "gene",
            unique_values_as_string,
        ),
        GeneID=(
            "GeneID",
            unique_values_as_string,
        ),
        transcript=(
            "transcript",
            unique_values_as_string,
        ),
    )
    .reset_index()
)


# Формирование основной таблицы no-exon target.

non_exonic_result = (
    no_exon_targets[
        TARGET_COLUMNS
    ]
    .merge(
        annotation_summary,
        on="p_name",
        how="left",
    )
)

non_exonic_result["exon_num"] = NA_VALUE

for column in [
    "feature_types",
    "gene",
    "GeneID",
    "transcript",
]:
    non_exonic_result[column] = (
        non_exonic_result[column]
        .fillna(NA_VALUE)
    )


# Расстояние до ближайшего exon.

closest_distance = (
    closest[
        ["a_name", "distance"]
    ]
    .rename(
        columns={
            "a_name": "p_name",
            "distance": "nearest_exon_distance",
        }
    )
)

if closest_distance["p_name"].duplicated().any():
    raise ValueError(
        "В closest_check.tsv для одного target "
        "присутствует более одного значения расстояния."
    )

non_exonic_result = non_exonic_result.merge(
    closest_distance,
    on="p_name",
    how="left",
)

missing_distance = (
    non_exonic_result["nearest_exon_distance"].isna()
)

if missing_distance.any():
    missing_targets = (
        non_exonic_result.loc[
            missing_distance,
            "p_name",
        ]
        .tolist()
    )

    raise ValueError(
        "Для следующих no-exon target "
        "не найдено расстояние до ближайшего exon: "
        f"{missing_targets}"
    )


# Определение типа RefSeq-аннотации.

non_exonic_result["annotation_status"] = (
    non_exonic_result["feature_types"]
    .apply(classify_annotation)
)


print("\nРаспределение по типу аннотации:")

print(
    non_exonic_result["annotation_status"]
    .value_counts()
    .to_string()
)


Распределение по типу аннотации:
annotation_status
region_only                  9
gene_or_transcript_region    8
RefSeq_match                 7
regulatory                   1


In [49]:
# Определение основного идентификатора для формирования final_id.
# Приоритет:
# 1. название гена
# 2. GeneID
# 3. метка unassigned

def get_annotation_label(row):
    if row["gene"] != NA_VALUE:
        return row["gene"]

    if row["GeneID"] != NA_VALUE:
        return f"GeneID{row['GeneID']}"

    return "unassigned"

non_exonic_result["final_id"] = (
    non_exonic_result.apply(
        make_non_exon_id,
        axis=1,
    )
)

NON_EXONIC_OUTPUT_COLUMNS = [
    "p_chrom",
    "p_start",
    "p_end",
    "p_name",
    "gene",
    "GeneID",
    "transcript",
    "exon_num",
    "annotation_status",
    "nearest_exon_distance",
    "final_id",
]

non_exonic_result = non_exonic_result[
    NON_EXONIC_OUTPUT_COLUMNS
].copy()


# Контроль уникальности target

if non_exonic_result["p_name"].duplicated().any():
    duplicated_targets = (
        non_exonic_result.loc[
            non_exonic_result["p_name"].duplicated(keep=False),
            "p_name",
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "В non-exonic результате обнаружены "
        f"дублирующиеся target: {duplicated_targets}"
    )

print(
    f"Уникальных non-exonic target: "
    f"{len(non_exonic_result):,}"
)

non_exonic_result.to_csv(
    OUTPUT_NO_EXON,
    sep="\t",
    index=False,
)

Уникальных non-exonic target: 25


In [50]:
# Объединение exonic и non-exonic аннотаций

full_table = pd.concat(
    [
        exonic_result,
        non_exonic_result,
    ],
    ignore_index=True,
)

full_table["p_start"] = (
    pd.to_numeric(
        full_table["p_start"],
        errors="raise",
    )
)

full_table = (
    full_table
    .sort_values(
        by=["p_chrom", "p_start"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Контроль количества target

if full_table["p_name"].duplicated().any():
    duplicated_targets = (
        full_table.loc[
            full_table["p_name"].duplicated(keep=False),
            "p_name",
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        "В итоговой таблице обнаружены "
        f"дублирующиеся target: {duplicated_targets}"
    )

print(
    f"Итоговых target: {len(full_table):,}"
)

print(
    f"  exonic:     {len(exonic_result):,}"
)

print(
    f"  non-exonic: {len(non_exonic_result):,}"
)

full_table.to_csv(
    OUTPUT_FINAL,
    sep="\t",
    index=False,
)

Итоговых target: 386
  exonic:     361
  non-exonic: 25


In [51]:
# Контроль соответствия исходной панели и итоговой аннотации.

original_panel = pd.read_csv(
    INPUT_PANEL,
    sep="\t",
    header=None,
    names=TARGET_COLUMNS + ["score", "attrs"],
    dtype=str,
)

original_ids = set(original_panel["p_name"])
final_ids = set(full_table["p_name"])

if original_ids != final_ids:
    missing = original_ids - final_ids
    extra = final_ids - original_ids

    raise ValueError(
        "Набор target в итоговой таблице не совпадает "
        "с исходной панелью.\n"
        f"Отсутствуют: {sorted(missing)}\n"
        f"Лишние: {sorted(extra)}"
    )

print(
    f"Проверка исходной панели: "
    f"{len(original_ids):,} target"
)

print(
    "Все target исходной панели присутствуют "
    "в итоговой таблице ровно по одному разу."
)

Проверка исходной панели: 386 target
Все target исходной панели присутствуют в итоговой таблице ровно по одному разу.


In [52]:
print("Итог")

print(
    f"Всего target: "
    f"{len(full_table):,}"
)

print(
    f"Уникальных генов: "
    f"{full_table.loc[
        full_table['gene'] != NA_VALUE,
        'gene'
    ].nunique():,}"
)

print("\nРаспределение по типу аннотации:")

print(
    full_table["annotation_status"]
    .value_counts()
    .to_string()
)


Итог
Всего target: 386
Уникальных генов: 33

Распределение по типу аннотации:
annotation_status
exon                         361
region_only                    9
gene_or_transcript_region      8
RefSeq_match                   7
regulatory                     1


In [53]:
gene_summary = (
    full_table.loc[
        full_table["gene"] != NA_VALUE,
        ["gene", "p_name"]
    ]
    .drop_duplicates()
    .groupby("gene")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="target_count")
)

gene_summary

,gene,target_count
0,INSR,48
1,ABCC8,41
2,GLIS3,34
3,HNF4A,25
4,HNF1A,21
5,BLK,20
6,GCK,20
7,CEL,16
8,KCNJ11,16
9,HNF1B,15


In [54]:
TARGET_ID = "AMPL7162509639"

print("=" * 80)
print(f"Поиск target: {TARGET_ID}")
print("=" * 80)

# ------------------------------------------------------------------
# 1. Все RefSeq-аннотации target
# ------------------------------------------------------------------

target_refseq = refseq[
    refseq["p_name"] == TARGET_ID
].copy()

print("\n=== RefSeq-аннотация ===")

if target_refseq.empty:
    print("Target не найден в panel_refseq_full.tsv")
else:
    print(
        target_refseq[
            [
                "p_chrom",
                "p_start",
                "p_end",
                "p_name",
                "g_chrom",
                "g_feature",
                "g_start",
                "g_end",
                "g_strand",
                "g_attrs",
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------------
# 2. Расстояние до ближайшего exon
# ------------------------------------------------------------------

target_closest = closest[
    closest["a_name"] == TARGET_ID
].copy()

print("\n=== Ближайший exon ===")

if target_closest.empty:
    print("Target не найден в closest_check.tsv")
else:
    print(
        target_closest.to_string(index=False)
    )


# ------------------------------------------------------------------
# 3. Итоговая запись после аннотации
# ------------------------------------------------------------------

target_result = full_table[
    full_table["p_name"] == TARGET_ID
].copy()

print("\n=== Итоговая аннотация ===")

if target_result.empty:
    print("Target не найден в итоговой таблице")
else:
    print(
        target_result.to_string(index=False)
    )

Поиск target: AMPL7162509639

=== RefSeq-аннотация ===
p_chrom  p_start    p_end         p_name g_chrom  g_feature  g_start     g_end g_strand                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              g_attrs
  chr10 14208466 14208737 AMPL7162509639   chr10       gene 13685706  14372923        -                                                                                                                                                                                                             